In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import (
    train_test_split, StratifiedKFold, GridSearchCV, cross_val_predict
)
from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    mean_squared_error, accuracy_score,
    precision_score, recall_score, f1_score
)

from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier

In [2]:
def compute_metrics(y_true, y_pred, label=""):
    mse  = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average='macro')
    rec  = recall_score(y_true, y_pred, average='macro')
    f1   = f1_score(y_true, y_pred, average='macro')
    return {
        "Set": label,
        "MSE": round(mse, 4),
        "RMSE": round(rmse, 4),
        "Accuracy": round(acc, 4),
        "Precision": round(prec, 4),
        "Recall": round(rec, 4),
        "F1": round(f1, 4)
    }

In [3]:
def normalize_target(price):
    if price > 7300:
        return 1.0
    elif price < 2160:
        return 0.0
    elif 2160 <= price <= 7300:
        return (price - 2160) / (7300 - 2160)
    else:
        return 0.0 

In [4]:
data = pd.read_csv('ostrava_housing.csv')
data['target_Normalized']=data['target'].apply(normalize_target)

# decreasing
columns_to_keep = [
    'ownership', 'energy_label', 'state', 'equipment', 'crime_index', 'quality_index'
]
for col in columns_to_keep:
    min_val = data[col].min()
    max_val = data[col].max()
    data[f"{col}_Normalized"] = 1- ((data[col] - min_val) / (max_val - min_val))

# increasing
columns_to_keep = [
    'area', 'rooms'
]
for col in columns_to_keep:
    min_val = data[col].min()
    max_val = data[col].max()
    data[f"{col}_Normalized"] = (data[col] - min_val) / (max_val - min_val)

preprocessed_data = data[[col for col in data.columns if col.endswith('_Normalized')]]
df_final = preprocessed_data.iloc[:,[2,3,4,5,6,1,8,7,0]]
df_final = df_final.rename(columns={'target_Normalized': 'target'})
df_final['target'] = df_final.apply(lambda row: 0 if (row['target']<0.5)else 1, axis=1)

X_train, X_test = train_test_split(df_final, test_size=0.2, random_state=42)

scaler = MinMaxScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=df_final.columns)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=df_final.columns)

X_train = X_train_scaled.iloc[:, :-1]
X_test = X_test_scaled.iloc[:, :-1]
y_train = X_train_scaled.iloc[:, -1]
y_test = X_test_scaled.iloc[:, -1]

In [5]:
X_train.describe()

,energy_label_Normalized,state_Normalized,equipment_Normalized,crime_index_Normalized,quality_index_Normalized,ownership_Normalized,rooms_Normalized,area_Normalized
count,1354.000000,1354.000000,1354.000000,1354.000000,1354.000000,1354.000000,1354.000000,1354.000000
mean,0.227770,0.552806,0.440177,0.539533,0.604874,0.738922,0.375000,0.130119
std,0.320564,0.425946,0.418665,0.249269,0.436769,0.438754,0.153597,0.073603
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,0.394834,0.000000,0.000000,0.250000,0.089189
50%,0.000000,0.500000,0.500000,0.642066,1.000000,1.000000,0.416667,0.121622
75%,0.600000,1.000000,1.000000,0.723247,1.000000,1.000000,0.500000,0.154054
max,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [6]:
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

models = {
    "Naive Bayes": {
        "pipeline": Pipeline([
            ("scaler", MinMaxScaler()),
            ("clf", GaussianNB())
        ]),
        "params": {
            "clf__var_smoothing": np.logspace(-9, -6, 7)
        }
    },

    "Logistic Regression": {
        "pipeline": Pipeline([
            ("scaler", MinMaxScaler()),
            ("clf", LogisticRegression(max_iter=100, random_state=2))
        ]),
        "params": {
            "clf__C": [0.001, 0.1, 1, 10]
        }
    },

    "Random Forest": {
        "pipeline": Pipeline([
            ("scaler", MinMaxScaler()),
            ("clf", RandomForestClassifier(random_state=2))
        ]),
        "params": {
            "clf__n_estimators": [50, 100, 200, 500],
            "clf__max_depth": [None, 2, 10]
        }
    },

    "Neural Network": {
        "pipeline": Pipeline([
            ("scaler", MinMaxScaler()),
            ("clf", MLPClassifier(max_iter=500, random_state=2))
        ]),
        "params": {
            "clf__hidden_layer_sizes": [(10,), (20,), (50,), (10,10), (50,50)],
            "clf__activation": ["relu", "tanh"],
            "clf__solver": ['adam', 'sgd']
        }
    }
}

In [7]:
all_results = []

for name, config in models.items():
    print(f"\n{'='*60}")
    print(f"  Model: {name}")
    print(f"{'='*60}")

    gs = GridSearchCV(
        estimator=config["pipeline"],
        param_grid=config["params"],
        cv=cv,
        scoring='accuracy',
        n_jobs=-1,
        verbose=0
    )
    gs.fit(X_train, y_train)

    best_model = gs.best_estimator_
    print(f"  The best params : {gs.best_params_}")
    print(f"  CV F1 (val. set)   : {gs.best_score_:.4f}")

    y_train_pred = cross_val_predict(best_model, X_train, y_train, cv=cv)
    train_metrics = compute_metrics(y_train, y_train_pred, label="Train (CV)")

    y_test_pred = best_model.predict(X_test)
    test_metrics = compute_metrics(y_test, y_test_pred, label="Test")

    for m in [train_metrics, test_metrics]:
        print(f"\n  [{m['Set']}]")
        for k, v in m.items():
            if k != "Set":
                print(f"    {k:<12}: {v}")

    all_results.append({"Model": name, **train_metrics})
    all_results.append({"Model": name, **test_metrics})


  Model: Naive Bayes
  The best params : {'clf__var_smoothing': 1e-09}
  CV F1 (val. set)   : 0.8279

  [Train (CV)]
    MSE         : 0.1721
    RMSE        : 0.4148
    Accuracy    : 0.8279
    Precision   : 0.7827
    Recall      : 0.8066
    F1          : 0.7926

  [Test]
    MSE         : 0.1593
    RMSE        : 0.3991
    Accuracy    : 0.8407
    Precision   : 0.7875
    Recall      : 0.7919
    F1          : 0.7897

  Model: Logistic Regression
  The best params : {'clf__C': 10}
  CV F1 (val. set)   : 0.8486

  [Train (CV)]
    MSE         : 0.1514
    RMSE        : 0.3891
    Accuracy    : 0.8486
    Precision   : 0.8209
    Recall      : 0.782
    F1          : 0.7977

  [Test]
    MSE         : 0.1534
    RMSE        : 0.3917
    Accuracy    : 0.8466
    Precision   : 0.8088
    Recall      : 0.7607
    F1          : 0.7795

  Model: Random Forest
  The best params : {'clf__max_depth': 10, 'clf__n_estimators': 500}
  CV F1 (val. set)   : 0.8915

  [Train (CV)]
    MSE      

In [8]:
print(f"\n\n{'='*60}")
print("Metrics")
print(f"{'='*60}")
df_results = pd.DataFrame(all_results)
df_results = df_results.set_index(["Model", "Set"])
print(df_results.to_string())



Metrics
                                   MSE    RMSE  Accuracy  Precision  Recall      F1
Model               Set                                                            
Naive Bayes         Train (CV)  0.1721  0.4148    0.8279     0.7827  0.8066  0.7926
                    Test        0.1593  0.3991    0.8407     0.7875  0.7919  0.7897
Logistic Regression Train (CV)  0.1514  0.3891    0.8486     0.8209  0.7820  0.7977
                    Test        0.1534  0.3917    0.8466     0.8088  0.7607  0.7795
Random Forest       Train (CV)  0.1086  0.3295    0.8914     0.8815  0.8368  0.8552
                    Test        0.1121  0.3348    0.8879     0.8703  0.8195  0.8404
Neural Network      Train (CV)  0.1396  0.3736    0.8604     0.8329  0.8045  0.8167
                    Test        0.1298  0.3603    0.8702     0.8378  0.8038  0.8185
